# Context Effect Recovery: Can Self-Attention Learn the True Mechanism?

This notebook investigates whether self-attention models, when trained on choice data, genuinely **recover** the mechanism that generated those choices, or merely predict well without capturing the real structure. We study this under two synthetic context-effect models, **LCL** (the slate average bends preferences) and **CDM** (pairwise item-item interaction), where the ground truth is known exactly by construction.

**This notebook closes the LCL half.** CDM follows in a separate notebook that reuses the same protocol.

Four architectures are compared on equal footing: a context-free MLP (a control), a mean-pool network (the simplest architecture with the correct structure), self-attention (the subject), and an oracle (the correct formula built in, the ceiling). Recovery is measured by one model-agnostic probe, validated on cases with a known answer before it is trusted on any trained model.

### Roadmap

| Part | What happens | Cost |
|---|---|---|
| A | Build the ground truth ($\theta$, $A$) | fast |
| B-D | Slates, simulated choices, train / val / eval_in / eval_shift splits | fast |
| E | The probe and the recovery metrics, validated analytically | fast |
| F | Models, training protocol, one fully inspected reference run | moderate |
| G | Five-seed reference study with confidence intervals | slow |
| H | Rank, sparsity and sample-size sweeps | slowest |

Everything slow saves its results to CSV files after every (configuration, seed). If a session dies, re-run the same cell and it resumes where it stopped.

## Setup

Imports, a consistent plot style, and the folder where result files are written (`/kaggle/working` on Kaggle, a local folder elsewhere).

In [ ]:
import os, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

# Where result files are written. Kaggle keeps /kaggle/working with a saved version;
# anywhere else we fall back to a local folder.
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "lcl_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("torch", torch.__version__, "| output folder:", OUT_DIR)

## Part A1: Building the ground truth ($\theta$ and $A$)

Before we can test whether any model recovers context effects, we need a true answer to test against: a baseline preference vector $\theta$ and the context-bending matrix $A$ used by LCL:

$$u_i(S) = (\theta + A\,y_S)^\top x_i, \qquad y_S = \frac{1}{|S|}\sum_{j \in S} x_j$$

$A$ is built at a controlled **rank** as $A = UV^\top$, so it has exactly `rank_A` independent directions, and is then rescaled so that $\|A\|_F$ equals `A_scale`. Fixing the size means every configuration carries the same total amount of context effect, so a difference in recovery cannot be blamed on a lucky large draw.

An optional **sparsity** knob zeroes a fraction of $A$'s entries at random (used only in Part H). Zeroing entries of a low-rank matrix generally raises its rank again, so rank and sparsity are studied in two separate sweeps and never combined.

Reference configuration: $d = 6$ features, rank 2, $\|A\|_F = 1$, slate size 5.

In [ ]:
def make_ground_truth(d_features, rank_A, A_scale, seed, sparsity=0.0):
    """
    theta : baseline preference vector, length d_features, ||theta|| = 1
    A     : the 'bending' matrix (d_features x d_features), ||A||_F = A_scale

    rank_A   : A = U @ V.T with U, V of shape (d, rank_A), so rank(A) = rank_A exactly.
    sparsity : fraction of A's entries forced to exactly zero (default 0 = dense).
               NOTE: masking a low-rank matrix usually raises its rank, so the
               rank sweep keeps sparsity = 0 and the sparsity sweep uses full rank.
    """
    rng = np.random.default_rng(seed)

    theta = rng.normal(0, 1, size=d_features)
    theta = theta / np.linalg.norm(theta)

    U = rng.normal(0, 1, size=(d_features, rank_A))
    V = rng.normal(0, 1, size=(d_features, rank_A))
    A_mat = U @ V.T

    if sparsity > 0:
        n_entries = d_features * d_features
        n_zero = int(round(sparsity * n_entries))
        flat_idx = rng.choice(n_entries, size=n_zero, replace=False)
        mask = np.ones(n_entries)
        mask[flat_idx] = 0
        A_mat = A_mat * mask.reshape(d_features, d_features)

    A_mat = A_mat / np.linalg.norm(A_mat, ord="fro")
    return theta, A_scale * A_mat


# --- configuration of the reference experiment ---
D_FEATURES = 6
RANK_A     = 2
A_SCALE    = 1.0
SEED       = 12345
SLATE_SIZE = 5
N_TRAIN    = 40000     # training sessions
N_VAL      = 5000      # validation sessions: used ONLY to pick the early-stopping checkpoint
N_EVAL     = 5000      # eval_in: held-out, same distribution as train (reporting only)
N_SHIFT    = 5000      # eval_shift: unusual slate compositions (reporting only)

theta, A = make_ground_truth(D_FEATURES, RANK_A, A_SCALE, SEED)

print("theta:", np.round(theta, 3))
print("theta norm:", round(np.linalg.norm(theta), 4))
print("A shape:", A.shape)
print("A singular values:", np.round(np.linalg.svd(A, compute_uv=False), 4))
print("A Frobenius norm:", round(np.linalg.norm(A, "fro"), 4))

## Part A2: Visualizing A

A picture of the ground-truth matrix, before we do anything else with
it. Red and blue show the sign of the bending effect for each
(item feature, slate-average feature) pair, and the printed numbers
give the exact values.

This is worth looking at closely: even though it's a 6x6 grid of 36
numbers, the underlying structure is really built from just 2
directions (rank_A=2) — so the pattern across rows and columns won't
look like independent random noise. Later, when we plot a model's
recovered A_hat next to this one, this is the reference picture every
comparison will be judged against.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))

vmax = np.abs(A).max()
im = ax.imshow(A, cmap="coolwarm", vmin=-vmax, vmax=vmax)

ax.set_title(f"Ground-truth context matrix A\n(rank = {RANK_A}, d = {D_FEATURES})")
ax.set_xlabel("feature index (in slate average $y_S$)")
ax.set_ylabel("feature index (of item $x_i$)")
ax.set_xticks(range(D_FEATURES))
ax.set_yticks(range(D_FEATURES))

for i in range(D_FEATURES):
    for j in range(D_FEATURES):
        ax.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, label="strength of bending")
plt.tight_layout()
plt.show()

## Side experiment: why $\|A\|_F$ is fixed

We build five versions of $A$ with the same rank and directions, differing only in overall size ($\|A\|_F$ from 0.2 to 50), and measure two things on simulated choices:

- **SNR**: variance of the context term $(A y_S)^\top x_i$ divided by variance of the baseline term $\theta^\top x_i$. How loud is the context effect? It should grow like $\|A\|_F^2$.
- **Choice entropy**: how undecided the choices are. The maximum is $\ln 5$, a fair coin over five items.

What to look for is what happens to entropy as the context term grows. If it falls toward 0, choices become near-deterministic and say little about *how strong* the effect is, even though the effect is loud. That is the reason we compare configurations at a controlled scale instead of whatever a random draw produces.

In [ ]:
def build_A_at_norm(d_features, rank_A, frob_norm, seed):
    """Same construction as make_ground_truth, but sets the Frobenius norm directly."""
    rng = np.random.default_rng(seed)
    U = rng.normal(0, 1, size=(d_features, rank_A))
    V = rng.normal(0, 1, size=(d_features, rank_A))
    A_mat = U @ V.T
    A_mat = A_mat / np.linalg.norm(A_mat, ord="fro")   # unit norm first
    return frob_norm * A_mat                            # then the target size


def quick_snr_and_entropy(theta, A_mat, n_sessions, slate_size, d_features, seed):
    """Simulates one batch of choices; returns SNR and mean choice entropy."""
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n_sessions, slate_size, d_features))
    y_S = X.mean(axis=1)

    base_term = np.einsum("d,nsd->ns", theta, X)
    ctx_term  = np.einsum("nd,nsd->ns", y_S @ A_mat.T, X)
    snr = ctx_term.var() / base_term.var()

    utilities = base_term + ctx_term
    z = utilities - utilities.max(axis=1, keepdims=True)
    probs = np.exp(z)
    probs /= probs.sum(axis=1, keepdims=True)

    entropy = -(probs * np.log(probs + 1e-12)).sum(axis=1).mean()
    return snr, entropy, np.log(slate_size)


norm_demo = []
for fn in [0.2, 1.0, 5.0, 20.0, 50.0]:
    A_test = build_A_at_norm(D_FEATURES, RANK_A, fn, seed=SEED)
    snr, ent, max_ent = quick_snr_and_entropy(
        theta, A_test, n_sessions=5000, slate_size=SLATE_SIZE,
        d_features=D_FEATURES, seed=1)
    norm_demo.append((fn, snr, ent, max_ent))
    print(f"||A||_F = {fn:>5} | SNR = {snr:9.3f} | entropy = {ent:.3f} "
          f"(max {max_ent:.3f}, {100 * ent / max_ent:.1f}% of max)")

In [ ]:
fig, ax1 = plt.subplots(figsize=(6.5, 4.5))

xs   = [r[0] for r in norm_demo]
snrs = [r[1] for r in norm_demo]
ents = [r[2] for r in norm_demo]

ax1.plot(xs, snrs, "o-", color="firebrick")
ax1.set_xlabel("$\\|A\\|_F$  (overall size of the context matrix)")
ax1.set_ylabel("SNR (context / baseline)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.set_xscale("log")
ax1.set_yscale("log")

ax2 = ax1.twinx()
ax2.plot(xs, ents, "s--", color="steelblue")
ax2.axhline(np.log(SLATE_SIZE), color="steelblue", alpha=0.3, linestyle=":")
ax2.set_ylabel("choice entropy (higher = more undecided)", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")
ax2.grid(False)

plt.title("A louder context effect vs. how informative the choices stay")
fig.tight_layout()
plt.show()

## Part B: Building slates, and watching context "concentrate"

Before we simulate any choices, we need to understand one property of
how slates are built: as slate size grows, the slate-average feature
vector y_S becomes less varied across different slates.

Why this matters: our whole ability to see A depends on y_S varying
from slate to slate. If every slate's y_S looks nearly the same, there
is nothing for A to visibly act differently on — the context signal
gets "washed out" even though A itself hasn't changed.

B1 builds slates and computes y_S. B2 plots how much y_S spreads out,
as a function of slate size.

In [ ]:
def draw_slates(n_sessions, slate_size, d_features, seed):
    """
    Draws n_sessions slates, each with slate_size items, each item
    described by d_features independent N(0,1) values.

    Returns:
        X   : shape (n_sessions, slate_size, d_features) — the items
        y_S : shape (n_sessions, d_features) — the per-slate average,
              i.e. what "the page looks like on average"
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n_sessions, slate_size, d_features))
    y_S = X.mean(axis=1)
    return X, y_S


# quick check on one slate size, just to see the shapes
X_check, y_S_check = draw_slates(n_sessions=5, slate_size=5,
                                  d_features=D_FEATURES, seed=1)
print("X shape:", X_check.shape)
print("y_S shape:", y_S_check.shape)
print("\nFirst slate's items:\n", np.round(X_check[0], 2))
print("\nFirst slate's average y_S:\n", np.round(y_S_check[0], 2))

In [ ]:
slate_sizes = [2, 5, 10, 20, 50, 100]
spreads = []

for m in slate_sizes:
    _, y_S_m = draw_slates(n_sessions=5000, slate_size=m, d_features=D_FEATURES, seed=1)
    spreads.append(y_S_m.std(axis=0).mean())   # average spread across features

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(slate_sizes, spreads, "o-", color="darkorange", label="measured")
ax.plot(slate_sizes, [1 / np.sqrt(m) for m in slate_sizes], "k--", alpha=0.6,
        label="theory: $1/\\sqrt{m}$")
ax.set_xlabel("slate size (m)")
ax.set_ylabel("spread of $y_S$ across slates\n(avg std per feature)")
ax.set_title("Bigger slates: less variety between pages")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.show()

print("slate_size : measured spread  (theory 1/sqrt(m))")
for m, s in zip(slate_sizes, spreads):
    print(f"  {m:>3} : {s:.4f}   ({1 / np.sqrt(m):.4f})")

## Part B - Summary: why slate size matters

**What we did:** built slates of different sizes ($m = 2$ to $100$) and measured how much the slate average $y_S$ varies across slates of each size.

**What we found:** the spread follows $1/\sqrt{m}$, from about 0.71 at $m=2$ down to 0.10 at $m=100$, exactly what averaging $m$ independent items predicts.

**Why this matters:** our whole method for detecting $A$ compares how preferences shift as the slate composition changes. If slates are large, every slate average looks the same, so there is little variety for $A$ to act on, even though $A$ itself has not changed. Slate size is therefore an identification lever, not a cosmetic setting.

**Going forward:** the reference configuration uses slate size 5.

## Part C: Turning slates into choices

So far we've only built pages (slates) — we haven't simulated anyone
actually choosing anything. Part C adds the choice mechanism: given a
slate and our ground-truth (theta, A), compute how attractive each item
is, turn that into a probability distribution over the slate, and
sample one item as "the choice."

C1 builds this step by step. C2 checks whether the resulting choices
are sensibly uncertain — not always obvious, not always a coin flip.

In [ ]:
def simulate_choices(theta, A, X, seed):
    """
    Given the ground truth (theta, A) and a batch of slates X,
    simulates one choice per session under the LCL model:

        eff_theta(S) = theta + A @ y_S
        u_i(S)       = eff_theta(S) . x_i
        P(i | S)     = softmax(u(S))
        choice       = sample from P(i | S)

    Returns y_S, utilities, probs, choices — all intermediate values,
    in case we need to inspect or plot any of them later.
    """
    rng = np.random.default_rng(seed)

    y_S = X.mean(axis=1)
    eff_theta = theta[None, :] + y_S @ A.T
    utilities = np.einsum("nd,nsd->ns", eff_theta, X)

    z = utilities - utilities.max(axis=1, keepdims=True)
    probs = np.exp(z)
    probs /= probs.sum(axis=1, keepdims=True)

    cdf = probs.cumsum(axis=1)
    draws = rng.random((X.shape[0], 1))
    choices = (draws > cdf).sum(axis=1)
    choices = np.clip(choices, 0, X.shape[1] - 1)

    return y_S, utilities, probs, choices


# quick check on the same 5-slate toy example
y_S_c, util_c, probs_c, choices_c = simulate_choices(theta, A, X_check, seed=1)
print("Probabilities for session 0:", np.round(probs_c[0], 3), "sum:", probs_c[0].sum())
print("Chosen items across all 5 sessions:", choices_c)

## Part C2: Are the simulated choices actually informative?

Now that we can simulate choices, a real question worth checking: are
they too easy (one item almost always wins, entropy near 0) or too hard
(every item is basically a coin flip, entropy near max)?

Either extreme is a problem. Too easy, and there's no meaningful
variation for the probe to detect. Too hard, and there's so much
randomness that any model — even the oracle — would struggle to see a
pattern. We want most sessions to land somewhere in between: genuinely
uncertain, but not pure noise.

In [ ]:
N_SESSIONS_DEMO = 5000
SLATE_SIZE = 5

X_demo, y_S_demo = draw_slates(N_SESSIONS_DEMO, SLATE_SIZE, D_FEATURES, seed=42)
_, util_demo, probs_demo, choices_demo = simulate_choices(theta, A, X_demo, seed=42)

entropy = -(probs_demo * np.log(probs_demo + 1e-12)).sum(axis=1)
max_entropy = np.log(SLATE_SIZE)

print(f"Max possible entropy (pure coin-flip over {SLATE_SIZE} items): {max_entropy:.3f}")
print(f"Mean entropy across {N_SESSIONS_DEMO} sessions: {entropy.mean():.3f}"
      f"  ({100*entropy.mean()/max_entropy:.1f}% of max)")
print(f"Min entropy (most 'obvious' session): {entropy.min():.3f}")
print(f"Max entropy (most 'undecided' session): {entropy.max():.3f}")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(entropy, bins=40, color="mediumseagreen", edgecolor="white")
ax.axvline(max_entropy, color="black", linestyle="--", label=f"max possible ({max_entropy:.2f})")
ax.set_xlabel("choice entropy per session")
ax.set_ylabel("number of sessions")
ax.set_title("How undecided are our simulated choices?")
ax.legend()
plt.tight_layout()
plt.show()

## Part C - Summary

The simulation pipeline is: slate, slate average $y_S$, bent preference $\theta + A y_S$, utilities, softmax probabilities, one sampled choice per session.

The entropy diagnostics show the reference configuration ($d=6$, rank 2, $\|A\|_F=1$, slate size 5) sits in a useful middle zone: choices are mostly undecided (mean entropy about 79% of the maximum) but with a real tail of decisive sessions. Neither pure noise nor near-deterministic, so the probe has something to detect.

## Part D: Assembling the data splits

We combine slate building (Part B) and choice simulation (Part C) into four splits, all generated from the **same** ground truth ($\theta$, $A$), so any difference between them is about slate composition, not about a different underlying world:

| split | role |
|---|---|
| `train` | the only data a model learns from |
| `val` | used **only** to pick each model's early-stopping checkpoint |
| `eval_in` | held-out, same distribution as train; reported, never used for any choice |
| `eval_shift` | deliberately unusual slate compositions; reported, never used for any choice |

Why a separate `val`: if the checkpoint were chosen on `eval_in` and then reported on `eval_in`, every model would look slightly better than it really is.

### D1: One shared draw for train / val / eval_in, plus a shifted split

`train`, `val` and `eval_in` come from **one** big draw sliced by index, so they are identically distributed by construction rather than by hope.

`eval_shift` uses ordinary items, drawn from the same $N(0,1)$ distribution, but keeps only the slates whose average $y_S$ is farthest from the origin: out of a pool 20 times larger, the most extreme compositions that plain random sampling rarely produces. Same ingredients, unusual recipe.

In [ ]:
def make_shifted_slates(n_shift, slate_size, d_features, seed, oversample=20):
    """Over-generate slates, keep the n_shift whose average y_S is farthest from 0."""
    rng = np.random.default_rng(seed)
    pool = rng.normal(0, 1, size=(n_shift * oversample, slate_size, d_features))
    norms = np.linalg.norm(pool.mean(axis=1), axis=1)
    keep = np.argsort(norms)[-n_shift:]
    return pool[keep]


def build_all_splits(theta_c, A_c, n_train, data_seed,
                     n_val=N_VAL, n_eval=N_EVAL, n_shift=N_SHIFT):
    """train / val / eval_in / eval_shift from ONE ground truth (theta_c, A_c)."""
    X_all, _ = draw_slates(n_train + n_val + n_eval, SLATE_SIZE, D_FEATURES, seed=data_seed)
    X_parts = {
        "train":      X_all[:n_train],
        "val":        X_all[n_train:n_train + n_val],
        "eval_in":    X_all[n_train + n_val:],
        "eval_shift": make_shifted_slates(n_shift, SLATE_SIZE, D_FEATURES,
                                          seed=data_seed + 1, oversample=20),
    }
    out = {}
    for k, (name, X_part) in enumerate(X_parts.items()):
        y_S, _, probs, choices = simulate_choices(theta_c, A_c, X_part, seed=data_seed * 10 + k)
        out[name] = dict(X=X_part, y_S=y_S, probs=probs, choices=choices)
    return out


# the reference splits: exactly the data that the reference run in Part F trains on
splits = build_all_splits(theta, A, N_TRAIN, data_seed=SEED + 1)

for name, s in splits.items():
    print(f"{name:<11} n = {s['X'].shape[0]:>6}   slate_size = {s['X'].shape[1]}   "
          f"features = {s['X'].shape[2]}")

### D2: Is `eval_shift` really unusual?

If the construction worked, `train`, `val` and `eval_in` overlap almost exactly in how unusual their slate averages are, while `eval_shift` sits clearly to the right.

In [ ]:
norms_train = np.linalg.norm(splits["train"]["y_S"], axis=1)
norms_eval_in = np.linalg.norm(splits["eval_in"]["y_S"], axis=1)
norms_eval_shift = np.linalg.norm(splits["eval_shift"]["y_S"], axis=1)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(norms_train, bins=40, alpha=0.6, label="train", color="steelblue", density=True)
ax.hist(norms_eval_in, bins=40, alpha=0.6, label="eval_in", color="seagreen", density=True)
ax.hist(norms_eval_shift, bins=40, alpha=0.6, label="eval_shift", color="firebrick", density=True)

ax.set_xlabel("$\\|y_S\\|$  (how unusual the page's overall composition is)")
ax.set_ylabel("density")
ax.set_title("How different is eval_shift from ordinary pages?")
ax.legend()
plt.tight_layout()
plt.show()

train_p99 = np.percentile(norms_train, 99)
frac_beyond = (norms_eval_shift > train_p99).mean()

print(f"train median ||y_S||:      {np.median(norms_train):.3f}")
print(f"eval_in median ||y_S||:    {np.median(norms_eval_in):.3f}")
print(f"eval_shift median ||y_S||: {np.median(norms_eval_shift):.3f}")
print(f"\ntrain's 99th percentile ||y_S||: {train_p99:.3f}")
print(f"fraction of eval_shift beyond that: {frac_beyond:.3f}")
print(f"val median ||y_S||:        {np.median(np.linalg.norm(splits['val']['y_S'], axis=1)):.3f}")

### Why eval_in and eval_shift are both necessary

eval_in and train share one distribution — the plot above should show
their histograms overlapping almost exactly. eval_in exists to answer
a narrow question: does the model generalize to unseen pages that look
like the ones it trained on? Good performance here is necessary, but
not enough, to trust that a model learned the real mechanism.

eval_shift is deliberately built from the rare tail of what ordinary
sampling could produce — the histogram should show it sitting well to
the right of train and eval_in, not overlapping them.

The reason we need both: a model could score just as well as another
on eval_in by memorizing patterns specific to typical pages, without
ever learning how preferences actually bend with context. Since
train and eval_in look alike, that shortcut would go undetected.
eval_shift is where it gets caught — a model that only memorized
should degrade there, while a model that learned the true rule
theta + A @ y_S should hold up, since that formula is just as valid on
an unusual page as an ordinary one.

So the pairing turns "did the model recover A" from an abstract
correlation number into a concrete, checkable claim: does its
predictive quality survive on pages unlike anything it trained on.

### Checking: did selecting for extreme y_S make eval_shift "too easy"?

eval_shift has higher ||y_S|| than train by construction. Bigger y_S
means a bigger context bend, which should make context effects more
visible — but if pushed too far, it can also make choices nearly
deterministic (as we saw earlier when ||A||_F got too large), which
would actually hurt rather than help.

We check this the same way we checked the base dataset in Part C2:
compare the entropy distributions of train vs. eval_shift directly.

In [ ]:
def compute_entropy(probs):
    return -(probs * np.log(probs + 1e-12)).sum(axis=1)

ent_train = compute_entropy(splits["train"]["probs"])
ent_shift = compute_entropy(splits["eval_shift"]["probs"])
max_ent = np.log(SLATE_SIZE)

print(f"train:      mean entropy = {ent_train.mean():.3f}  ({100*ent_train.mean()/max_ent:.1f}% of max)")
print(f"eval_shift: mean entropy = {ent_shift.mean():.3f}  ({100*ent_shift.mean()/max_ent:.1f}% of max)")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(ent_train, bins=40, alpha=0.6, label="train", color="steelblue", density=True)
ax.hist(ent_shift, bins=40, alpha=0.6, label="eval_shift", color="firebrick", density=True)
ax.axvline(max_ent, color="black", linestyle="--", label=f"max possible ({max_ent:.2f})")
ax.set_xlabel("choice entropy")
ax.set_ylabel("density")
ax.set_title("Is eval_shift still informative, or did it become too easy?")
ax.legend()
plt.tight_layout()
plt.show()

## Part E: The probe estimator — recovering A from any model's scores

So far everything has been about building data, not measuring a model.
Part E builds the *instrument* we'll use to ask: "given a trained
model, how much of the true A did it actually learn?"

### How this differs from our earlier evaluation metrics (Metric A / Metric C)

Earlier work (Metric A, Metric C) asked a **behavioral** question: does
the model act context-sensitively? Concretely, swap one item for
another and watch how the log-ratio of choice probabilities shifts.
Metric C in particular was designed to survive softmax saturation,
which had made Metric A's raw slope readings unreliable (the same true
effect could read as 0.51 or 2.65 depending purely on which probability
region was probed).

What we're building now asks a **structural** question instead: can we
reconstruct the actual matrix A, entry by entry, and compare it
directly to the ground truth? This is strictly harder, and it matters
because of a gap we already found on CDM — direct weight readout said
recovery was ~0, while Metric C (behavioral) said 0.737. That gap is
real, but it leaves something unresolved: does a high Metric C score
mean the model found the *right* mechanism, or just *some* mechanism
that happens to produce similar swap behavior? Full matrix
reconstruction is what lets us answer that more precisely.

### The core idea

Place two fixed probe items (a, b) into many different slates, and
watch how the model's score gap between them shifts as the surrounding
filler items change. Under LCL, letting $\delta = x_a - x_b$:

$$\Delta(S) = \theta^\top \delta \;+\; y_S^\top (A^\top \delta)$$

This is ordinary linear regression: regressing the score gap on $y_S$
recovers one "shadow" of A — a single projection, seen from the angle
of that one probe pair. To reconstruct the *full* matrix, we repeat
this with several different probe pairs, each viewed from a different
angle, and solve a joint linear system for A itself.

We'll build this in three small steps:
  - **E1** — construct one probe pair and the slates around it *(the probe pair stays fixed across trials while the fillers vary: the property this method depends on)*
  - **E2** — regress score gap on $y_S$, for one probe pair
  - **E3** — repeat across several probe pairs and solve for $\hat A$

In [ ]:
def make_one_probe_pair(d_features, seed):
    """
    A single probe pair (a, b): two fixed items we'll reuse across many
    slates. Drawn from the same N(0,1) distribution as ordinary items,
    so the model is never asked to score something out-of-distribution.
    """
    rng = np.random.default_rng(seed)
    x_a = rng.normal(0, 1, size=d_features)
    x_b = rng.normal(0, 1, size=d_features)
    return x_a, x_b


def build_slates_around_pair(x_a, x_b, n_trials, slate_size, d_features, seed):
    """
    n_trials slates, each with the SAME probe pair at positions 0 and 1,
    and fresh random filler items everywhere else. Since a and b never
    change, any change in the score gap across trials must come from
    the fillers -- which is exactly what moves y_S.
    """
    rng = np.random.default_rng(seed)
    n_fill = slate_size - 2
    fillers = rng.normal(0, 1, size=(n_trials, n_fill, d_features))
    a_block = np.broadcast_to(x_a, (n_trials, 1, d_features))
    b_block = np.broadcast_to(x_b, (n_trials, 1, d_features))
    return np.concatenate([a_block, b_block, fillers], axis=1)


# --- quick check with one probe pair ---
x_a, x_b = make_one_probe_pair(D_FEATURES, seed=99)
X_probe = build_slates_around_pair(x_a, x_b, n_trials=5, slate_size=SLATE_SIZE,
                                    d_features=D_FEATURES, seed=1)

print("Probe item a:", np.round(x_a, 3))
print("Probe item b:", np.round(x_b, 3))
print("\nOne example slate (a and b should be rows 0 and 1, identical across trials):")
print(np.round(X_probe[0], 3))
print("\nAnother trial's slate (rows 0,1 same as above, rest different):")
print(np.round(X_probe[1], 3))

### E2: Regress the score gap on y_S — for ONE probe pair

Now we scale up from 5 toy trials to a proper number, so the
regression actually has enough data to fit anything meaningful.

For this step we use the TRUE scoring function (the generator's own
formula) rather than a trained model — same reasoning as Part C's
validation: check the estimator's arithmetic works perfectly under
zero-noise, known-answer conditions, before ever trusting it on
something noisier.

If this works, the regression's slope should come out equal to
$g = A^\top \delta$ almost exactly — computable directly and independently,
giving us a known answer to check against.

In [ ]:
def score_fn_true(X):
    """The generator's own scoring rule, exposed as score_fn(X) -> scores.
    Same interface any trained model will eventually plug into."""
    y_S = X.mean(axis=1)
    eff_theta = theta[None, :] + y_S @ A.T
    return np.einsum("nd,nsd->ns", eff_theta, X)


# --- one probe pair, many trials this time ---
N_TRIALS = 2000

x_a, x_b = make_one_probe_pair(D_FEATURES, seed=99)
X_probe = build_slates_around_pair(x_a, x_b, n_trials=N_TRIALS, slate_size=SLATE_SIZE,
                                    d_features=D_FEATURES, seed=1)

y_S_probe = X_probe.mean(axis=1)
scores = score_fn_true(X_probe)
delta_scores = scores[:, 0] - scores[:, 1]   # score(a) - score(b), per trial

# --- regression: [1, y_S] -> delta_scores ---
design = np.concatenate([np.ones((N_TRIALS, 1)), y_S_probe], axis=1)
coef, *_ = np.linalg.lstsq(design, delta_scores, rcond=None)
intercept_hat = coef[0]
slope_hat = coef[1:]           # this should equal A^T @ delta

# --- known answer, computed directly, to check against ---
delta = x_a - x_b
intercept_true = theta @ delta
slope_true = A.T @ delta

print("intercept: fitted =", round(intercept_hat, 4), " true =", round(intercept_true, 4))
print("\nslope (g = A^T delta):")
print("  fitted:", np.round(slope_hat, 4))
print("  true:  ", np.round(slope_true, 4))
print("\nmax abs difference:", np.max(np.abs(slope_hat - slope_true)))

# R^2, to check the fit quality
pred = design @ coef
ss_res = ((delta_scores - pred) ** 2).sum()
ss_tot = ((delta_scores - delta_scores.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print("\nR^2 of the regression:", round(r2, 6))

### E3: Repeat across several probe pairs, and solve for the full A

One probe pair gives one projection of A — a single "shadow," as seen
from one angle. To reconstruct the full 6x6 matrix, we need several
probe pairs, each giving a shadow from a different angle, then solve
for A jointly.

Concretely: for K probe pairs, each with direction $\delta_k = x_a^{(k)} - x_b^{(k)}$,
we get K slope vectors $g_k = A^\top \delta_k$. Stacking these into
matrices $D$ (rows $\delta_k$) and $G$ (rows $g_k$):

$$G = D A$$

and we solve for A via least squares. From the earlier analytic check
(back when we validated this same math with K=6, 12, 24), we know
K = 2d gives good conditioning — stable inversion, not overly sensitive
to noise. With d=6, that means K=12 probe pairs.

In [ ]:
def estimate_A_from_scores(score_fn, K, n_trials, slate_size, d_features, seed):
    """
    Runs the E1+E2 procedure across K probe pairs, then solves for the
    full matrix A. score_fn is model-agnostic -- it only ever sees
    slates in, scores out.
    """
    rng = np.random.default_rng(seed)

    D_rows = []
    G_rows = []

    for k in range(K):
        x_a_k = rng.normal(0, 1, size=d_features)
        x_b_k = rng.normal(0, 1, size=d_features)
        delta_k = x_a_k - x_b_k

        X_k = build_slates_around_pair(x_a_k, x_b_k, n_trials, slate_size,
                                        d_features, seed=rng.integers(1e9))
        y_S_k = X_k.mean(axis=1)
        scores_k = score_fn(X_k)
        delta_scores_k = scores_k[:, 0] - scores_k[:, 1]

        design = np.concatenate([np.ones((n_trials, 1)), y_S_k], axis=1)
        coef, *_ = np.linalg.lstsq(design, delta_scores_k, rcond=None)

        D_rows.append(delta_k)
        G_rows.append(coef[1:])

    D_mat = np.array(D_rows)   # (K, d)
    G_mat = np.array(G_rows)   # (K, d)

    A_hat, *_ = np.linalg.lstsq(D_mat, G_mat, rcond=None)
    return A_hat, D_mat, G_mat


K = 2 * D_FEATURES   # = 12
A_hat, D_mat, G_mat = estimate_A_from_scores(
    score_fn_true, K=K, n_trials=2000, slate_size=SLATE_SIZE,
    d_features=D_FEATURES, seed=7
)

print("Recovered A_hat:")
print(np.round(A_hat, 4))
print("\nTrue A:")
print(np.round(A, 4))
print("\nMax abs difference:", np.max(np.abs(A_hat - A)))
print("Condition number of D:", round(np.linalg.cond(D_mat), 2))

### E4: Scoring a recovered matrix

A recovered $\hat A$ is compared with the true $A$ using four numbers, each answering a different question. Both matrices are treated as flat vectors:

- **cosine** $= \dfrac{\langle \hat A, A\rangle}{\|\hat A\|\,\|A\|}$: did it point the right *direction*? Blind to magnitude.
- **scale_ratio** $= \|\hat A\| / \|A\|$: did it get the *strength* right? 1 is correct, 0 means no effect found.
- **rel_frob** $= \|\hat A - A\| / \|A\|$: overall error, direction and magnitude together. 0 is perfect, and 1 is what answering $\hat A = 0$ scores.
- **zero_leak**: defined only when $A$ has exact zeros (the sparsity sweep). Mean $|\hat A|$ over the true-zero entries divided by mean $|\hat A|$ over the true-nonzero entries. 0 means nothing is assigned to interactions that do not exist.

A worked example you can check by hand: $A = \begin{pmatrix}2&0\\0&0\end{pmatrix}$ and $\hat A = \begin{pmatrix}1&0\\0&0\end{pmatrix}$ (right direction, half the strength). Expected: cosine 1.0, scale_ratio 0.5, rel_frob 0.5, zero_leak 0.

In [ ]:
def recovery_metrics(A_hat, A_true):
    """cosine (direction), rel_frob (overall error), scale_ratio (strength),
    zero_leak (mass placed on entries that are truly zero; NaN if A has no zeros)."""
    a, b = A_hat.ravel(), A_true.ravel()
    cosine = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
    rel_frob = float(np.linalg.norm(A_hat - A_true) / np.linalg.norm(A_true))
    scale_ratio = float(np.linalg.norm(A_hat) / np.linalg.norm(A_true))

    zero_mask = np.isclose(A_true, 0.0)
    if zero_mask.any() and (~zero_mask).any():
        zero_leak = float(np.abs(A_hat[zero_mask]).mean() /
                          (np.abs(A_hat[~zero_mask]).mean() + 1e-12))
    else:
        zero_leak = float("nan")

    return dict(cosine=cosine, rel_frob=rel_frob, scale_ratio=scale_ratio, zero_leak=zero_leak)


# the hand-checkable example from the markdown above
A_toy     = np.array([[2.0, 0.0], [0.0, 0.0]])
A_hat_toy = np.array([[1.0, 0.0], [0.0, 0.0]])
print({k: round(v, 4) for k, v in recovery_metrics(A_hat_toy, A_toy).items()})

### Part E — Summary

We built the probe estimator in three steps:
  - E1: confirmed slates around a probe pair correctly hold (a, b)
    fixed while varying fillers
  - E2: confirmed a single probe pair's regression exactly recovers
    its "shadow" of A (one projection, g = A^T delta)
  - E3: confirmed stacking K=12 probe pairs and solving the joint
    system G = DA exactly recovers the FULL matrix A

All three passed at floating-point precision (differences ~1e-16),
confirming the estimator's arithmetic is correct.

**Important scope check:** this only validates the *math*. We fed in
the generator's own scoring function — zero noise, and a model that
literally *is* A. It does not yet tell us whether this same procedure
works on a model that stores context through some other mechanism
entirely (a neural net's weights, an attention pattern). That is a
separate, harder question, and it's the next real test — not a
formality, since this is exactly where CDM's direct-weight-readout
approach silently failed before (reporting ~0 when the true answer
was 0.737).

## Part F: Model zoo and training protocol

| model | sees $x_i$ | sees the slate | knows the LCL formula | role |
|---|---|---|---|---|
| `context_free_mlp` | yes | no | no | **control**: the probe must read $\hat A \approx 0$ |
| `mean_pool_network` | yes | only via $y_S$, handed to it | no | simplest architecture with the right inputs |
| `self_attention` | yes | via learned attention | no | **the subject** (dropout 0) |
| `self_attention_do0.1` | yes | via learned attention | no | ablation: the common dropout default |
| `oracle` | yes | yes | **yes** | ceiling: the shape is given, only the numbers are learned |

**Protocol, identical for every model:**

1. Full-batch Adam, learning rate 0.01, cross-entropy over the slate.
2. Early stopping on the separate `val` split. The checkpoint with the lowest `val` NLL is kept, and training stops after `PATIENCE` epochs without improvement.
3. **Every measurement is taken in eval mode** (NLL and the probe readout), so dropout is off. Dropout randomly disables units while training; left on at measurement time it adds noise to every number.
4. **Parameter matching:** the context-free MLP and the mean-pool network are widened to about the same parameter count as self-attention, so a gap between them cannot be blamed on size. The oracle has only 42 parameters by design.
5. Self-attention is reported with dropout 0 (architecture only) and, as an ablation, with dropout 0.1.
6. Several seeds. In every seed the ground truth $A$, the data and the initialisation are all redrawn, and within a seed all models see the same $A$ and the same data, so differences are paired.

In [ ]:
# ---------------- training configuration ----------------
LR         = 1e-2
MAX_EPOCHS = 300
PATIENCE   = 60          # stop after this many epochs without val improvement; None = always run MAX_EPOCHS
PIPELINE_TAG = f"lcl-v3-pat{PATIENCE}"   # stored with every saved row, so stale CSVs are never mixed in


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def eval_nll(model, X_eval, choices_eval):
    """NLL on held-out data, with the model forced into eval mode (dropout off)
    and restored to its previous mode afterwards."""
    was_training = model.training
    model.eval()
    X_t = torch.tensor(X_eval, dtype=torch.float32)
    y_t = torch.tensor(choices_eval, dtype=torch.long)
    with torch.no_grad():
        loss = nn.CrossEntropyLoss()(model(X_t), y_t).item()
    model.train(was_training)
    return loss


def score_fn_from_torch_model(model):
    """Wraps a trained model as score_fn(X) -> scores for the probe,
    always in eval mode so dropout cannot add noise to the readout."""
    def f(X):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            out = model(torch.tensor(X, dtype=torch.float32)).numpy()
        model.train(was_training)
        return out
    return f


def train_with_early_stopping(model, X_train, choices_train, X_val, choices_val,
                              n_epochs=MAX_EPOCHS, lr=LR, patience=PATIENCE,
                              verbose=True, print_every=50):
    """Returns (train_losses, val_losses, best_val_nll); the model is left holding
    the weights from its best-validation epoch."""
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(choices_train, dtype=torch.long)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    best_val, best_state, best_epoch = float("inf"), None, 0

    for epoch in range(n_epochs):
        model.train()                              # the training step always runs in train mode
        optimizer.zero_grad()
        loss = loss_fn(model(X_t), y_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        v = eval_nll(model, X_val, choices_val)    # eval mode inside
        val_losses.append(v)
        if v < best_val:
            best_val, best_epoch = v, epoch
            best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch % print_every == 0 or epoch == n_epochs - 1):
            print(f"epoch {epoch:>4}: train={loss.item():.4f}  val={v:.4f}  (best val so far {best_val:.4f})")
        if patience is not None and epoch - best_epoch >= patience:
            if verbose:
                print(f"early stop at epoch {epoch} (best epoch {best_epoch})")
            break

    model.load_state_dict(best_state)
    return train_losses, val_losses, best_val


print("training config:", dict(LR=LR, MAX_EPOCHS=MAX_EPOCHS, PATIENCE=PATIENCE), "| tag:", PIPELINE_TAG)

### F1: The architectures, and the parameter matching

Self-attention is the reference size (one encoder layer, `d_model=32`, 2 heads). The context-free MLP and the mean-pool network get the hidden width whose parameter count lands closest to it. Dropout has no parameters, so the dropout setting does not change the count.

In [ ]:
class ContextFreeMLP(nn.Module):
    """Scores ONE item at a time. Never sees the slate or y_S."""
    def __init__(self, d_features, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, X):
        # X: (n_sessions, slate_size, d_features) -> scores: (n_sessions, slate_size)
        return self.net(X).squeeze(-1)


class MeanPoolNetwork(nn.Module):
    """Sees x_i and the slate mean y_S (concatenated), then an ordinary MLP.
    The averaging is forced; what to do with it is learned."""
    def __init__(self, d_features, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * d_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, X):
        y_S = X.mean(dim=1, keepdim=True).expand(-1, X.shape[1], -1)
        return self.net(torch.cat([X, y_S], dim=-1)).squeeze(-1)


class SelfAttentionModel(nn.Module):
    """Every item attends to every other item in its slate; nothing forces averaging."""
    def __init__(self, d_features, d_model=32, n_heads=2, n_layers=1, dropout=0.0):
        super().__init__()
        self.embed = nn.Linear(d_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.score_head = nn.Linear(d_model, 1)

    def forward(self, X):
        return self.score_head(self.encoder(self.embed(X))).squeeze(-1)


class LCLOracle(nn.Module):
    """theta and A are free parameters, but the functional FORM is the true LCL form."""
    def __init__(self, d_features):
        super().__init__()
        self.theta = nn.Parameter(torch.randn(d_features) * 0.1)
        self.A = nn.Parameter(torch.randn(d_features, d_features) * 0.1)

    def forward(self, X):
        y_S = X.mean(dim=1)
        eff_theta = self.theta.unsqueeze(0) + y_S @ self.A.T
        return torch.einsum("nd,nsd->ns", eff_theta, X)


def find_matching_hidden_width(model_class, target_params, d_features, hidden_range=range(4, 300)):
    """Hidden width whose parameter count lands closest to target_params."""
    best = (None, float("inf"), None)
    for hidden in hidden_range:
        n = count_params(model_class(d_features, hidden))
        if abs(n - target_params) < best[1]:
            best = (hidden, abs(n - target_params), n)
    return best[0], best[2]


n_params_attn = count_params(SelfAttentionModel(D_FEATURES))
hidden_mlp, n_mlp = find_matching_hidden_width(ContextFreeMLP, n_params_attn, D_FEATURES)
hidden_mp,  n_mp  = find_matching_hidden_width(MeanPoolNetwork, n_params_attn, D_FEATURES)


def model_specs():
    return [
        ("context_free_mlp",     ContextFreeMLP,     dict(d_features=D_FEATURES, hidden=hidden_mlp)),
        ("mean_pool_network",    MeanPoolNetwork,    dict(d_features=D_FEATURES, hidden=hidden_mp)),
        ("self_attention",       SelfAttentionModel, dict(d_features=D_FEATURES, dropout=0.0)),
        ("self_attention_do0.1", SelfAttentionModel, dict(d_features=D_FEATURES, dropout=0.1)),
        ("oracle",               LCLOracle,          dict(d_features=D_FEATURES)),
    ]


MODEL_COLORS = {"context_free_mlp": "gray", "mean_pool_network": "steelblue",
                "self_attention": "firebrick", "self_attention_do0.1": "orange",
                "oracle": "seagreen"}

print(f"{'model':<24}{'parameters':>11}")
print("-" * 35)
for name, cls, kwargs in model_specs():
    print(f"{name:<24}{count_params(cls(**kwargs)):>11}")
print(f"\nmatched widths: context_free_mlp hidden={hidden_mlp}, mean_pool_network hidden={hidden_mp}")

### F2: One configuration, one seed, all models

`run_one` is the whole validated pipeline for a single (rank, sparsity, seed): build the ground truth and the four splits, train every model with early stopping on `val`, probe each for $\hat A$, and record NLL on `eval_in` and `eval_shift`.

`shift_benefit = NLL(eval_in) - NLL(eval_shift)`: how much more a model gains when context matters more (the shifted slates carry a stronger context signal). A model that cannot see context cannot gain from it.

`run_configs_resumable` repeats `run_one` over configurations and seeds, appending to a CSV after every (configuration, seed) and skipping anything already saved.

In [ ]:
def run_one(rank_A, sparsity, seed, n_train=N_TRAIN, models=None, n_epochs=MAX_EPOCHS,
            lr=LR, patience=PATIENCE, return_artifacts=False):
    """The full pipeline for ONE configuration and ONE seed.
    The seed drives the ground-truth draw, the data draw and the model initialisation."""
    theta_c, A_c = make_ground_truth(D_FEATURES, rank_A, A_SCALE, seed, sparsity=sparsity)
    actual_rank = int(np.sum(np.linalg.svd(A_c, compute_uv=False) > 1e-6))
    splits_c = build_all_splits(theta_c, A_c, n_train, data_seed=seed + 1)

    rows, artifacts = [], {}
    for name, cls, kwargs in model_specs():
        if models is not None and name not in models:
            continue
        torch.manual_seed(seed)
        model = cls(**kwargs)
        tr, va, best_val = train_with_early_stopping(
            model, splits_c["train"]["X"], splits_c["train"]["choices"],
            splits_c["val"]["X"], splits_c["val"]["choices"],
            n_epochs=n_epochs, lr=lr, patience=patience, verbose=False)

        A_hat_c, *_ = estimate_A_from_scores(
            score_fn_from_torch_model(model), K=2 * D_FEATURES, n_trials=2000,
            slate_size=SLATE_SIZE, d_features=D_FEATURES, seed=7)
        mets = recovery_metrics(A_hat_c, A_c)

        nll_in = eval_nll(model, splits_c["eval_in"]["X"], splits_c["eval_in"]["choices"])
        nll_sh = eval_nll(model, splits_c["eval_shift"]["X"], splits_c["eval_shift"]["choices"])
        rows.append(dict(seed=seed, rank_A=rank_A, sparsity=sparsity, actual_rank=actual_rank,
                         n_train=n_train, model=name, params=count_params(model),
                         epochs_run=len(va), best_epoch=int(np.argmin(va)), val_nll=best_val,
                         eval_in_nll=nll_in, eval_shift_nll=nll_sh,
                         shift_benefit=nll_in - nll_sh, **mets))
        artifacts[name] = dict(model=model, A_hat=A_hat_c, train_losses=tr, val_losses=va)

    if return_artifacts:
        return rows, dict(models=artifacts, A_true=A_c, theta=theta_c, splits=splits_c)
    return rows


# ---------------- saving and resuming ----------------
def save_rows(rows, path):
    df = pd.DataFrame(rows)
    df["pipeline"] = PIPELINE_TAG
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def _empty_results():
    return pd.DataFrame({"seed": pd.Series(dtype="int64"), "rank_A": pd.Series(dtype="int64"),
                         "sparsity": pd.Series(dtype="float64"), "n_train": pd.Series(dtype="int64"),
                         "model": pd.Series(dtype="object")})


def load_results(path):
    """Rows saved by the CURRENT pipeline version only (so stale files cannot leak in)."""
    if not os.path.exists(path):
        return _empty_results()
    df = pd.read_csv(path)
    if "pipeline" not in df.columns:
        return _empty_results()
    return df[df["pipeline"] == PIPELINE_TAG].reset_index(drop=True)


def run_configs_resumable(configs, seeds, path, models=None, n_train=N_TRAIN):
    """configs: list of (rank_A, sparsity). Skips every (config, seed) already in `path`."""
    prev = load_results(path)
    done = set(zip(prev.rank_A, prev.sparsity.round(6), prev.n_train, prev.seed))
    skipped = 0
    for rank_A, sparsity in configs:
        for s in seeds:
            if (rank_A, round(sparsity, 6), n_train, s) in done:
                skipped += 1
                continue
            t0 = time.time()
            save_rows(run_one(rank_A, sparsity, s, n_train=n_train, models=models), path)
            print(f"saved: rank={rank_A} sparsity={sparsity} n_train={n_train} seed={s}"
                  f"  ({time.time() - t0:.0f}s)")
    if skipped:
        print(f"({skipped} runs were already saved and were skipped)")
    return load_results(path)

### F3: The reference run (seed 12345), fully inspected

One complete run at the reference configuration, with the trained models kept so we can look inside: learning curves, and the recovered matrices next to the truth. Its rows are saved to the reference CSV, and the five-seed study in Part G reuses them.

Before trusting anything else, three things must hold in this table:

1. `context_free_mlp` reads cosine $\approx 0$ and scale_ratio $\approx 0$. The probe reports nothing when there is nothing to find.
2. `oracle` has scale_ratio close to 1 and the smallest rel_frob.
3. No model's `best_epoch` sits at the end of its budget. If one does, its budget was too short and its "best" checkpoint is not real.

In [ ]:
REF_CSV = os.path.join(OUT_DIR, "lcl_reference.csv")

t0 = time.time()
ref_rows, ref_art = run_one(RANK_A, 0.0, SEED, return_artifacts=True)
print(f"reference run took {time.time() - t0:.0f}s\n")

if SEED not in set(load_results(REF_CSV).seed):
    save_rows(ref_rows, REF_CSV)

ref_df = pd.DataFrame(ref_rows)
cols = ["model", "params", "epochs_run", "best_epoch", "val_nll", "eval_in_nll",
        "eval_shift_nll", "shift_benefit", "cosine", "rel_frob", "scale_ratio"]
print(ref_df[cols].round(4).to_string(index=False))

In [ ]:
# Learning curves: when did each model stop improving on val?
# (For the dropout model the train curve is measured with dropout ON, so it sits
#  higher than it otherwise would; the val curve is always measured in eval mode.)
art = ref_art["models"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, (name, a) in zip(axes.ravel(), art.items()):
    ax.plot(a["train_losses"], label="train NLL", color="steelblue")
    ax.plot(a["val_losses"], label="val NLL", color="firebrick")
    best = int(np.argmin(a["val_losses"]))
    ax.axvline(best, color="gray", linestyle="--", label=f"best epoch = {best}")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.legend()
axes.ravel()[-1].axis("off")
axes[0, 0].set_ylabel("NLL")
axes[1, 0].set_ylabel("NLL")
plt.tight_layout()
plt.show()

In [ ]:
# The recovered matrices next to the truth, on one shared colour scale.
panels = [("true A", ref_art["A_true"])] + [(n, a["A_hat"]) for n, a in ref_art["models"].items()]
vmax = np.abs(ref_art["A_true"]).max()

fig, axes = plt.subplots(2, 3, figsize=(13, 8.5), constrained_layout=True)
for ax, (title, M) in zip(axes.ravel(), panels):
    im = ax.imshow(M, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(D_FEATURES))
    ax.set_yticks(range(D_FEATURES))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="strength of bending")
plt.show()

## Part G: Five seeds at the reference configuration

One seed cannot separate a real difference from luck: with a single run we have already seen a gap of 0.12 in scale_ratio shrink to about 0.06 once more seeds were averaged. Here every seed redraws the ground truth $A$, the data and the initialisation, and we compare models **within** a seed (paired differences) with 95% confidence intervals.

The cell below resumes automatically, so if the session dies, run it again. To add seeds, extend `SEEDS`.

In [ ]:
SEEDS = [SEED, 101, 202, 303, 404]
ms_df = run_configs_resumable([(RANK_A, 0.0)], SEEDS, REF_CSV)

metrics = ["eval_in_nll", "shift_benefit", "cosine", "rel_frob", "scale_ratio"]
print(ms_df.groupby("model")[metrics].agg(["mean", "std"]).round(4).to_string())
print("\nseeds per model:", ms_df.groupby("model").seed.nunique().to_dict())

In [ ]:
from scipy import stats

# 1. Was the training budget long enough?
print(ms_df.groupby("model")[["epochs_run", "best_epoch"]].agg(["min", "median", "max"]).to_string())
print(f"\nruns that used the full {MAX_EPOCHS}-epoch budget: {int((ms_df.epochs_run >= MAX_EPOCHS).sum())}\n")


# 2. Paired differences with 95% confidence intervals
def paired_ci(df, a, b, metric):
    pa = df[df.model == a].set_index("seed")[metric]
    pb = df[df.model == b].set_index("seed")[metric]
    d = (pa - pb).dropna()
    n = len(d)
    half = stats.t.ppf(0.975, n - 1) * d.std() / np.sqrt(n)
    return d.mean(), d.mean() - half, d.mean() + half, n


pairs = [("self_attention", "mean_pool_network"),
         ("self_attention", "self_attention_do0.1"),
         ("self_attention", "oracle"),
         ("mean_pool_network", "oracle")]
print(f"{'A minus B':<46}{'metric':<14}{'mean':>8}{'95% CI':>22}   n")
for a, b in pairs:
    for metric in ["rel_frob", "scale_ratio", "eval_in_nll", "shift_benefit"]:
        m, lo, hi, n = paired_ci(ms_df, a, b, metric)
        print(f"{a + ' - ' + b:<46}{metric:<14}{m:>+8.4f}   [{lo:+.4f}, {hi:+.4f}]   {n}")


# 3. Among context-aware models, does lower recovery error go with more benefit on eval_shift?
def within_seed_corr(df):
    d = df.copy()
    for col in ["rel_frob", "shift_benefit"]:
        d[col] = d[col] - d.groupby("seed")[col].transform("mean")
    return np.corrcoef(d.rel_frob, d.shift_benefit)[0, 1]


ca = ms_df[ms_df.model != "context_free_mlp"]
print(f"\nwithin-seed corr(rel_frob, shift_benefit): "
      f"context-aware models {within_seed_corr(ca):+.3f} | "
      f"without the oracle {within_seed_corr(ca[ca.model != 'oracle']):+.3f}")

In [ ]:
order = ["context_free_mlp", "mean_pool_network", "self_attention", "self_attention_do0.1", "oracle"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (metric, title) in zip(axes, [("rel_frob", "recovery error (rel_frob), lower is better"),
                                      ("scale_ratio", "recovered strength (scale_ratio), 1 is correct"),
                                      ("shift_benefit", "benefit on eval_shift (shift_benefit)")]):
    sns.stripplot(data=ms_df, x="model", y=metric, order=order, ax=ax, color="black", alpha=0.6, size=6)
    means = ms_df.groupby("model")[metric].mean().reindex(order)
    ax.scatter(np.arange(len(order)), means.values, marker="_", s=1200, color="firebrick", zorder=3)
    if metric == "scale_ratio":
        ax.axhline(1.0, color="gray", linestyle="--")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Each dot is one seed; the red bar is the mean over seeds", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Within each seed, subtract the seed's mean so only differences BETWEEN models remain.
d = ca.copy()
for col in ["rel_frob", "shift_benefit"]:
    d[col + "_dev"] = d[col] - d.groupby("seed")[col].transform("mean")

fig, ax = plt.subplots(figsize=(6.2, 5))
for m, g in d.groupby("model"):
    ax.scatter(g.rel_frob_dev, g.shift_benefit_dev, label=m, color=MODEL_COLORS[m], s=60, alpha=0.8)
ax.axhline(0, color="gray", linewidth=1)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("rel_frob minus the seed's mean   (lower = better recovery)")
ax.set_ylabel("shift_benefit minus the seed's mean")
ax.set_title("Does better recovery buy more benefit when context matters more?", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part H: Sweeps

Three separate sweeps, each varying one thing while everything else stays at the reference value:

| sweep | varies | held fixed |
|---|---|---|
| **rank** | rank of $A$ from 1 to 6 | sparsity 0 |
| **sparsity** | fraction of zeroed entries of $A$ from 0 to 0.8 | rank 6 (full rank, so sparsity does not fight a rank constraint) |
| **sample size** | training sessions from 2,500 to 40,000 | the reference $A$ |

Rank and sparsity are not varied together because zeroing entries of a low-rank matrix generally raises its rank. The sparsity sweep also reports `zero_leak`, the mass a model puts on interactions that do not exist.

The sample-size sweep is an addition to the rank and sparsity sweeps your supervisor asked for. An early one-seed check suggested that data scarcity may be where architectures separate; this sweep tests that properly, and it can be skipped.

To keep the cost down, the sweeps use four models (without the dropout ablation) and three seeds. Three seeds give only rough error bars, so read small differences with care. Each sweep is the slowest part of the notebook. It saves after every (configuration, seed), so you can stop it and re-run the same cell to continue.

In [ ]:
SWEEP_MODELS = ["context_free_mlp", "mean_pool_network", "self_attention", "oracle"]
SWEEP_SEEDS  = [101, 202, 303]


def plot_sweep(df, xcol, xlabel, metrics, models=SWEEP_MODELS, logx=False):
    """One panel per metric; one line per model; error bars are the std over seeds."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(5.4 * len(metrics), 4.6))
    axes = np.atleast_1d(axes)
    for ax, (metric, title) in zip(axes, metrics):
        for m in models:
            g = df[df.model == m].groupby(xcol)[metric].agg(["mean", "std"]).reset_index()
            ax.errorbar(g[xcol], g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3,
                        label=m, color=MODEL_COLORS[m])
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(xlabel)
        if logx:
            ax.set_xscale("log")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def sweep_table(df, xcol, metric):
    return df.pivot_table(index=xcol, columns="model", values=metric, aggfunc=["mean", "std"]).round(3)

### H1: Rank sweep (sparsity = 0, rank 1 to 6)

In [ ]:
RANK_CSV = os.path.join(OUT_DIR, "lcl_rank_sweep.csv")
rank_df = run_configs_resumable([(r, 0.0) for r in range(1, 7)], SWEEP_SEEDS, RANK_CSV,
                                models=SWEEP_MODELS)

print("actual rank of A per requested rank:", rank_df.groupby("rank_A").actual_rank.unique().to_dict())
print("\nmean rel_frob:\n", sweep_table(rank_df, "rank_A", "rel_frob")["mean"].to_string())
print("\nmean scale_ratio:\n", sweep_table(rank_df, "rank_A", "scale_ratio")["mean"].to_string())

plot_sweep(rank_df, "rank_A", "rank of A",
           [("rel_frob", "recovery error (rel_frob)"),
            ("scale_ratio", "recovered strength (scale_ratio)"),
            ("shift_benefit", "benefit on eval_shift")])

### H2: Sparsity sweep (rank 6, sparsity 0 to 0.8)

`actual_rank` is printed because masking changes the rank. `zero_leak` is undefined at sparsity 0 (there are no zeros), so the curves start at 0.2.

In [ ]:
SPARSITY_CSV = os.path.join(OUT_DIR, "lcl_sparsity_sweep.csv")
sparsity_df = run_configs_resumable([(6, s) for s in [0.0, 0.2, 0.4, 0.6, 0.8]], SWEEP_SEEDS,
                                    SPARSITY_CSV, models=SWEEP_MODELS)

print("actual rank of A per sparsity:", sparsity_df.groupby("sparsity").actual_rank.unique().to_dict())
print("\nmean rel_frob:\n", sweep_table(sparsity_df, "sparsity", "rel_frob")["mean"].to_string())
print("\nmean zero_leak:\n", sweep_table(sparsity_df, "sparsity", "zero_leak")["mean"].to_string())

plot_sweep(sparsity_df, "sparsity", "fraction of zeroed entries in A",
           [("rel_frob", "recovery error (rel_frob)"),
            ("scale_ratio", "recovered strength (scale_ratio)"),
            ("zero_leak", "zero_leak (mass on true zeros), lower is better"),
            ("shift_benefit", "benefit on eval_shift")])

### H3: Sample-size sweep (optional)

Reference $A$ (rank 2, sparsity 0), training sessions from 2,500 to 40,000. `val`, `eval_in` and `eval_shift` stay at 5,000 sessions each, so only the amount of training data changes.

In [ ]:
SAMPLE_SIZES = [2500, 5000, 10000, 20000, 40000]
SS_CSV = os.path.join(OUT_DIR, "lcl_sample_size_sweep.csv")

for n in SAMPLE_SIZES:
    ss_df = run_configs_resumable([(RANK_A, 0.0)], SWEEP_SEEDS, SS_CSV, models=SWEEP_MODELS, n_train=n)

print("mean rel_frob:\n", sweep_table(ss_df, "n_train", "rel_frob")["mean"].to_string())
print("\nmean scale_ratio:\n", sweep_table(ss_df, "n_train", "scale_ratio")["mean"].to_string())

plot_sweep(ss_df, "n_train", "training sessions",
           [("rel_frob", "recovery error (rel_frob)"),
            ("scale_ratio", "recovered strength (scale_ratio)"),
            ("shift_benefit", "benefit on eval_shift")], logx=True)

In [ ]:
# Everything that was saved, so nothing is lost when the session ends.
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".csv"):
        print(f"{f:<32} {len(pd.read_csv(os.path.join(OUT_DIR, f))):>5} rows")